# Cryo-EM benchmark

In [1]:
import csv
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from ctypes.util import find_library
from importlib.util import find_spec
from pathlib import Path

ROOT = Path.cwd()
WORK = Path("/workspace/cryo-benchmark-work")
PHENIX_INSTALLER = ROOT / "Phenix-2.2.1-6174-Linux-x86_64.sh"
PHENIX_ENV = WORK / "phenix" / "phenix_env.sh"
SOURCES = WORK / "sources"
ENVS = WORK / "envs"
RUNS = WORK / "runs"
METHODS = ("MICA", "ModelAngelo", "EModelX", "CryoAtom", "EMProt")
CPU_WORKERS = 4

targets = {}
for name, emdb_id, resolution in [("22XP", "68757", 3.20), ("32MC", "58994", 2.95)]:
    folder = ROOT / "data" / "targets" / name
    job = f"fold_{name.lower()}_1"
    target = {
        "map": folder / "inputs" / f"emd_{emdb_id}.map",
        "fasta": folder / "inputs" / f"rcsb_pdb_{name}.fasta",
        "af2": next(folder.glob("predictions/af2/*/*_unrelaxed_rank_001_*.pdb")),
        "af3": folder / "predictions" / "af3" / job / f"{job}_model_0.cif",
        "reference": folder / "reference" / f"{name}.cif",
    }
    for file in target.values():
        if not file.is_file():
            raise FileNotFoundError(file)
    targets[name] = dict(target, emdb_id=emdb_id, contour=0.06, resolution=resolution)

In [2]:
os.environ.update(
    MPLBACKEND="Agg",
    TORCH_HOME=str(WORK / "torch-cache"),
    CONDA_CHANNEL_PRIORITY="flexible",
    CONDA_PKGS_DIRS=str(WORK / "conda-pkgs"),
    UV_CACHE_DIR=str(WORK / "uv-cache"),
)
os.environ.pop("UV_SYSTEM_PYTHON", None)


def run(args, log=None, **options):
    if log is not None:
        options.update(stdout=log, stderr=subprocess.STDOUT)
    return subprocess.run([str(arg) for arg in args], check=True, text=True, **options)


def checkout(name, url, revision):
    repo = SOURCES / name
    if repo.exists():
        run(["git", "-C", repo, "checkout", "--quiet", "--detach", revision])
    else:
        with tempfile.TemporaryDirectory(dir=SOURCES) as staging:
            clone = Path(staging) / name
            run(["git", "clone", "--no-checkout", url, clone])
            run(["git", "-C", clone, "checkout", "--quiet", "--detach", revision])
            clone.rename(repo)
    return repo


def unpack(url, filename, destination):
    archive = WORK / filename
    run(["curl", "-fL", "--retry", "3", url, "-o", archive])
    shutil.unpack_archive(archive, destination)
    archive.unlink()


def uv_command():
    if shutil.which("uv"):
        return ["uv", "--no-config"]
    if find_spec("uv") is None:
        run([sys.executable, "-m", "pip", "install", "uv"])
    return [sys.executable, "-m", "uv", "--no-config"]


def uv_environment(name, cuda, torch, packages):
    prefix = ENVS / name
    python = prefix / "bin" / "python"
    if not (prefix / ".ready").exists():
        uv = uv_command()
        if not python.exists():
            run([*uv, "venv", "--python", "3.11", prefix])
        pip = [*uv, "pip", "install", "--python", python]
        run([*pip, "--index-url", f"https://download.pytorch.org/whl/{cuda}", *torch])
        run([*pip, *packages])
        (prefix / ".ready").touch()
    return prefix / "bin"


def conda_environment(repo, file):
    conda = Path(shutil.which("conda") or WORK / "miniforge" / "bin" / "conda")
    if not conda.exists():
        installer = WORK / "miniforge.sh"
        run(["curl", "-fL", "--retry", "3",
             "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
             "-o", installer])
        run(["bash", installer, "-b", "-p", conda.parent.parent])
        installer.unlink()

    prefix = ENVS / repo.name
    ready = prefix / ".ready"
    if not ready.exists():
        action = "update" if (prefix / "conda-meta" / "history").exists() else "create"
        run([conda, "env", action, "-p", prefix, "-f", repo / file])
        ready.touch()
    return [conda, "run", "-p", prefix, "--no-capture-output"]


def install_phenix():
    if not PHENIX_ENV.is_file():
        run(["bash", PHENIX_INSTALLER, "-b", "-p", PHENIX_ENV.parent])


def phenix(*args):
    return ["bash", "-c", 'source "$1" && shift && exec "$@"', "phenix", PHENIX_ENV, *args]


def benchmark(method, build):
    for name, target in targets.items():
        folder = RUNS / name / method
        folder.mkdir(parents=True, exist_ok=True)
        record = folder / "result.json"
        if record.exists():
            model = json.loads(record.read_text()).get("model")
            if model and Path(model).is_file():
                continue

        attempt = Path(tempfile.mkdtemp(prefix="run-", dir=folder))
        log_path = attempt / "run.log"
        print(f"{name} {method} → {log_path}", flush=True)
        start = time.perf_counter()
        result = {"log": str(log_path)}
        try:
            with log_path.open("w") as log:
                model = build(target, attempt, log)
            if not model.is_file() or model.stat().st_size == 0:
                raise FileNotFoundError(model)
            result.update(status="ok", model=str(model))
        except (OSError, subprocess.CalledProcessError) as error:
            result.update(status="failed", error=str(error))
            print(error)
        result["runtime_seconds"] = round(time.perf_counter() - start, 1)
        record.write_text(json.dumps(result))

In [3]:
for folder in (SOURCES, ENVS, RUNS):
    folder.mkdir(parents=True, exist_ok=True)

packages = [name for name in ("git", "curl", "make", "g++") if not shutil.which(name)]
if not find_library("GL"):
    packages.append("libgl1")
if not find_library("gfortran"):
    packages.append("libgfortran5")
if packages:
    run(["apt-get", "update"])
    run(["apt-get", "install", "-y", *packages])

In [ ]:
# MICA (Option B)
install_phenix()
mica = checkout("MICA", "https://github.com/jianlin-cheng/MICA.git", "726d1bb")
mica_bin = uv_environment("MICA", "cu124", ["torch==2.5.1"], [
    "numpy==1.26.4", "scipy==1.13.1", "biopython==1.83", "open3d-cpu==0.19.0",
    "superpose3d==1.1.1", "networkx==3.4.2", "rotary-embedding-torch==0.8.9",
    "mrcfile", "psutil", "tqdm", "einops", "natsort", "matplotlib", "requests",
])
if not (mica / "trained_models" / "MICA_best_model.pth").is_file():
    unpack("https://zenodo.org/records/15756654/files/trained_models.tar.gz?download=1",
           "mica.tar.gz", mica)


def build_mica(target, attempt, log):
    af3 = attempt / "AF3_results"
    job = af3 / target["af3"].parent.name
    job.mkdir(parents=True)
    shutil.copy2(target["af3"], job)
    out = attempt / "model"
    map_file, fasta, resolution = (target[key] for key in ("map", "fasta", "resolution"))
    python = [mica_bin / "python", "-u"]
    mica_env = dict(os.environ, PATH=f"{mica_bin}{os.pathsep}{os.environ['PATH']}")
    options = dict(cwd=mica, log=log, env=mica_env)

    run([*python, "utils/process_AF3_results.py", "-f", fasta, "-a", af3, "-d", "cuda:0"], **options)
    run([*python, "utils/dock_in_map.py", "-m", map_file, "-f", fasta, "-a", af3,
         "-c", target["contour"], "-r", resolution, "-x", PHENIX_ENV], **options)
    docked = attempt / f"{attempt.name}_af3_docked.pdb"
    if not docked.is_file():
        raise FileNotFoundError(docked)
    run([*python, "run.py", "-m", map_file, "-f", fasta, "-a", af3, "-r", resolution, "-o", out,
         "--device", "cuda:0", "--run_pulchra", "-p", "modules/pulchra304/src/pulchra",
         "--no_parallel", "--mul_proc_num", CPU_WORKERS], **options)

    raw = out / f"{target['emdb_id']}_{fasta.stem}_MICA_all_atom_model.pdb"
    run(phenix("phenix.real_space_refine", raw, map_file, mica / "modules" / "phenix.eff",
               f"resolution={resolution}", f"nproc={CPU_WORKERS}"), cwd=out, log=log)
    refined = sorted(out.glob(f"{raw.stem}_real_space_refined*"))
    for file in refined:
        if file.suffix in (".pdb", ".cif"):
            return file
    raise FileNotFoundError(f"Missing refined model in {out}")


benchmark("MICA", build_mica)

22XP MICA → /workspace/cryo-benchmark-work/runs/22XP/MICA/run-ug77n3t0/run.log


In [4]:
# ModelAngelo
angelo = checkout("ModelAngelo", "https://github.com/3dem/model-angelo.git", "994945b")
angelo_bin = uv_environment("ModelAngelo", "cu126", ["torch==2.9.1", "torchvision==0.24.1"], [angelo])
angelo_cli = angelo_bin / "model_angelo"
run([angelo_cli, "setup_weights", "--bundle-name", "nucleotides"])


def build_angelo(target, attempt, log):
    out = attempt / "model"
    run([angelo_cli, "build", "-v", target["map"], "-pf", target["fasta"], "-o", out, "-d", "0"],
        cwd=angelo, log=log)
    return out / "model.cif"


benchmark("ModelAngelo", build_angelo)

Using Python 3.11.18 environment at: /workspace/cryo-benchmark-work/envs/ModelAngelo
Resolved 28 packages in 11ms
Prepared 9 packages in 22.42s
Installed 28 packages in 2.18s
 + filelock==4.2.4
 + fsspec==2026.9.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.4.6
 + nvidia-cublas-cu12==12.6.4.1
 + nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.6.77
 + nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cufft-cu12==11.3.0.4
 + nvidia-cufile-cu12==1.11.1.6
 + nvidia-curand-cu12==10.3.7.77
 + nvidia-cusolver-cu12==11.7.1.2
 + nvidia-cusparse-cu12==12.5.4.2
 + nvidia-cusparselt-cu12==0.7.1
 + nvidia-nccl-cu12==2.27.5
 + nvidia-nvjitlink-cu12==12.6.85
 + nvidia-nvshmem-cu12==3.3.20
 + nvidia-nvtx-cu12==12.6.77
 + pillow==13.0.0
 + sympy==1.14.0
 + torch==2.9.1+cu126
 + torchvision==0.24.1+cu126
 + triton==3.5.1
 + typing-extensions==4.16.0
Using Python 3.11.18 environment at: /workspace/cryo-benchmark-work/envs/M

Please make sure you have set the environment variable TORCH_HOME 
to a suitable directory, visible to all relevant users!
Setting up bundle with name: nucleotides for the first time.


100%|██████████| 1.98G/1.98G [05:44<00:00, 6.17MB/s]


Checking checksum of downloaded file.
Checksum of downloaded file matches expected checksum.
Bundle nucleotides successfully installed.
Setting up language model bundle with name: esm1b_t33_650M_UR50S for the first time.


100%|██████████| 7.29G/7.29G [02:07<00:00, 61.6MB/s]
100%|██████████| 3.10k/3.10k [00:00<00:00, 2.20MB/s]


Successful!
Path to model is /workspace/cryo-benchmark-work/torch-cache/hub/checkpoints/model_angelo_v1.0/nucleotides
22XP ModelAngelo → /workspace/cryo-benchmark-work/runs/22XP/ModelAngelo/run-z4_zs1yf/run.log
32MC ModelAngelo → /workspace/cryo-benchmark-work/runs/32MC/ModelAngelo/run-qt7qys1a/run.log


In [5]:
# EModelX(+AF)
emodelx = checkout("EModelX", "https://github.com/biomed-AI/EModelX.git", "53713cf")
emodelx_run = conda_environment(emodelx, "EModelX.yml")
weights = emodelx / "models"
weights.mkdir(exist_ok=True)
for atom, file_id in [
    ("CA", "1n8yiVr4oRxsY4ii2a05hcx5gI3eIybA1"),
    ("BB", "1GbmezR4tVCYaALZ330TL4vk84kGDQD3V"),
    ("AA", "1evghUiUcb3mFiCQ2-Gjez9AtA2w6Pyir"),
]:
    file = weights / f"best_{atom}_model.ckpt"
    if not file.is_file():
        partial = file.with_suffix(".part")
        run([*uv_command(), "tool", "run", "gdown", file_id, "-O", partial])
        partial.replace(file)


def build_emodelx(target, attempt, log):
    templates = attempt / "templates"
    fasta = target["fasta"]
    fasta_id = fasta.read_text().splitlines()[0][1:].split("|")[0]
    template = templates / fasta_id / "ranked_0.pdb"
    template.parent.mkdir(parents=True)
    shutil.copy2(target["af2"], template)
    out = attempt / "model"
    run([*emodelx_run, "python", "-u", "run.py", "--protocol", "temp_flex",
         "--EM_map", target["map"], "--fasta", fasta, "--template_dir", templates,
         "--output_dir", out, "--run_pulchra", "--pulchra_path", "modules/pulchra304/src/pulchra",
         "--mul_proc_num", CPU_WORKERS], cwd=emodelx, log=log)
    return out / f"{fasta.stem}_EModelX(+AF)_all_atom_model.pdb"


benchmark("EModelX", build_emodelx)

Cloning into '/workspace/cryo-benchmark-work/sources/tmptt4dcqjs/EModelX'...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  122M  100  122M    0     0  61.5M      0  0:00:01  0:00:01 --:--:-- 86.1M


PREFIX=/workspace/cryo-benchmark-work/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.10.23-hbd8a1cb_0.conda
Extracting libgomp-16.2.0-he0feb66_7.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-9_cp314.conda
Extracting tzdata-2026e-h5d301ff_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_103.conda
Extracting libgcc-16.2.0-ha9f2e26_7.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.9-hebe6cf0_0.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.9.0-hd2095e1_0.conda
Extracting libffi-3.7.0-h81df57d_1.conda
Extracting libiconv-1.18-h0cb94f2_3.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extracting libpython-3.14

warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.


Linking _openmp_mutex-4.5-20_gnu
Linking zstd-1.5.7-hb78ec9c_7
Linking ld_impl_linux-64-2.46.1-default_hbd61a6d_103
Linking libgcc-16.2.0-ha9f2e26_7
Linking bzip2-1.0.8-hda65f42_10
Linking c-ares-1.34.9-hebe6cf0_0
Linking keyutils-1.6.3-h7cc23a3_1
Linking libev-4.33-h280c20c_3
Linking libexpat-2.9.0-hd2095e1_0
Linking libffi-3.7.0-h81df57d_1
Linking libiconv-1.18-h0cb94f2_3
Linking liblzma-5.8.3-hb03c661_1
Linking libmpdec-4.0.0-hb03c661_2
Linking libpython-3.14.10-hd51c5da_100_cp314
Linking libstdcxx-16.2.0-h934c35e_7
Linking libuuid-2.42.4-hcfc3c73_1
Linking lzo-2.10-hebe6cf0_1003
Linking ncurses-6.6-hdb14827_1
Linking openssl-3.6.5-h781a0a9_0
Linking reproc-14.2.8.post0-hee9eb32_0
Linking tk-8.6.13-noxft_h1df4ec4_4
Linking cpp-expected-1.3.1-h171cf75_1
Linking fmt-12.1.0-h76c4fd7_1
Linking icu-78.3-py310h44b86e0_2
Linking libedit-3.1.20260512-pl5321h373387f_0
Linking libmsgpack-c-6.1.0-h54a6638_7
Linking libnghttp2-1.71.0-hc5e5977_0
Linking libsolv-0.7.41-h72ddc62_0
Linking libssh2-

WARNING conda.conda_pypi.main:notify_conda_pypi_tip(158): 
  Did you know? You can install many PyPI packages with conda
  using conda-pypi. Get started:
    https://bit.ly/4xXYt0B



done
Installing pip dependencies: ...working... Ran pip subprocess with arguments:
['/workspace/cryo-benchmark-work/envs/EModelX/bin/python', '-m', 'pip', 'install', '-U', '-r', '/workspace/cryo-benchmark-work/sources/EModelX/condaenv.8nm0zm85.requirements.txt', '--exists-action=b']
Pip subprocess output:
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
    Preparing wheel metadata: started
    Preparing wheel metadata: finished with status 'done'
  Using cached cffi-1.17.1-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (446 kB)
  Using cached pycparser-2.23-py3-none-any.whl (118 kB)
  Using cached cython-3.2.9-cp38-cp38-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (3.5 MB)
  Created wheel for freesasa: filename=freesasa-2.2.1-cp38-cp38-linux_x86_64.whl size=1030146 sha256=2e4f7b42ed426ed

Installed 12 packages in 33ms
Downloading...
From (original): https://drive.google.com/uc?id=1n8yiVr4oRxsY4ii2a05hcx5gI3eIybA1
From (redirected): https://drive.google.com/uc?id=1n8yiVr4oRxsY4ii2a05hcx5gI3eIybA1&confirm=t&uuid=f18dbf67-ccf9-413f-a6c3-646e31dee7e8
To: /workspace/cryo-benchmark-work/sources/EModelX/models/best_CA_model.part
100%|██████████| 424M/424M [00:09<00:00, 43.6MB/s] 
Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1GbmezR4tVCYaALZ330TL4vk84kGDQD3V
From (redirected): https://drive.google.com/uc?id=1GbmezR4tVCYaALZ330TL4vk84kGDQD3V&confirm=t&uuid=f0989aa0-06db-41b8-8d24-4b8a982785a1
To: /workspace/cryo-benchmark-work/sources/EModelX/models/best_BB_model.part
100%|██████████| 424M/424M [00:08<00:00, 47.8MB/s] 
Using cookies from /root/.cache/gdown/cookies.txt
Downloading...
From (original): https://drive.google.com/uc?id=1evghUiUcb3mFiCQ2-Gjez9AtA2w6Pyir
From (redirected): https://drive.google.com/uc?id

22XP EModelX → /workspace/cryo-benchmark-work/runs/22XP/EModelX/run-9zbe3m4s/run.log


100%|██████████| 424M/424M [00:09<00:00, 42.7MB/s]


Command '['/workspace/cryo-benchmark-work/miniforge/bin/conda', 'run', '-p', '/workspace/cryo-benchmark-work/envs/EModelX', '--no-capture-output', 'python', '-u', 'run.py', '--protocol', 'temp_flex', '--EM_map', '/workspace/mica-benchmark/data/targets/22XP/inputs/emd_68757.map', '--fasta', '/workspace/mica-benchmark/data/targets/22XP/inputs/rcsb_pdb_22XP.fasta', '--template_dir', '/workspace/cryo-benchmark-work/runs/22XP/EModelX/run-9zbe3m4s/templates', '--output_dir', '/workspace/cryo-benchmark-work/runs/22XP/EModelX/run-9zbe3m4s/model', '--run_pulchra', '--pulchra_path', 'modules/pulchra304/src/pulchra', '--mul_proc_num', '4']' returned non-zero exit status 1.
32MC EModelX → /workspace/cryo-benchmark-work/runs/32MC/EModelX/run-dxvjep7c/run.log
Command '['/workspace/cryo-benchmark-work/miniforge/bin/conda', 'run', '-p', '/workspace/cryo-benchmark-work/envs/EModelX', '--no-capture-output', 'python', '-u', 'run.py', '--protocol', 'temp_flex', '--EM_map', '/workspace/mica-benchmark/data/

In [6]:
# CryoAtom v1.0
cryoatom = checkout("CryoAtom", "https://github.com/YangLab-SDU/CryoAtom.git", "v1.0.0")
cryoatom_run = conda_environment(cryoatom, "linux.yml")
checkpoints = cryoatom / "CryoAtom" / "checkpoint"
if not all((checkpoints / name).is_file() for name in ("CryNet.pth", "SimpleUnet.pth")):
    unpack("https://yanglab.qd.sdu.edu.cn/CryoAtom/download/checkpoints_v1.0.zip",
           "cryoatom.zip", cryoatom)
if not (ENVS / "CryoAtom" / "bin" / "cryoatom").is_file():
    run([*cryoatom_run, "python", "-m", "pip", "install", "."], cwd=cryoatom)


def build_cryoatom(target, attempt, log):
    out = attempt / "model"
    run([*cryoatom_run, "cryoatom", "build", "-v", target["map"], "-s", target["fasta"],
         "-o", out, "-d", "cuda:0"], cwd=cryoatom, log=log)
    return out / "model.cif"


benchmark("CryoAtom", build_cryoatom)

Cloning into '/workspace/cryo-benchmark-work/sources/tmpetu9z9er/CryoAtom'...


Channels:
 - pytorch
 - nvidia
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /workspace/cryo-benchmark-work/envs/CryoAtom

  added / updated specs:
    - cudatoolkit=11.3.1
    - mkl=2023.0.0
    - numpy=1.21.5
    - pip
    - python=3.8
    - pytorch==1.11.0=py3.8_cuda11.3_cudnn8.2.0_0
    - torchaudio==0.11.0=py38_cu113
    - torchvision==0.12.0=py38_cu113
    - tqdm


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |       8_kmp_llvm           8 KB  conda-forge
    aom-3.4.0                  |       h27087fc_1         2.7 MB  conda-forge
    blas-1.0                   |              mkl           1 KB  conda-forge
    brotli-python-1.1.0        |   py38h17151c0_1         343 KB  conda-forge
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates

WARNING conda.conda_pypi.main:notify_conda_pypi_tip(158): 
  Did you know? You can install many PyPI packages with conda
  using conda-pypi. Get started:
    https://bit.ly/4xXYt0B



done
Installing pip dependencies: ...working... Ran pip subprocess with arguments:
['/workspace/cryo-benchmark-work/envs/CryoAtom/bin/python', '-m', 'pip', 'install', '-U', '-r', '/workspace/cryo-benchmark-work/sources/CryoAtom/condaenv.hiq70y5_.requirements.txt', '--exists-action=b']
Pip subprocess output:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 131.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 193.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 208.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 208.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 8.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 179.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 111.7 MB/s eta 0:00:00

done
#
# To activate this environment, use
#
#     $ conda activate /workspace/cryo-benchmark-work/envs/CryoAtom
#
# To d

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  618M  100  618M    0     0  7715k      0  0:01:22  0:01:22 --:--:-- 5819k


Processing /workspace/cryo-benchmark-work/sources/CryoAtom
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for CryoAtom: filename=CryoAtom-1.0.0-py3-none-any.whl size=649861980 sha256=457d2bfddb7bd668f9a22e227bed17c9535f7f349b681eb073b18b4c8f1cbffa
  Stored in directory: /tmp/pip-ephem-wheel-cache-z6bs1ez3/wheels/68/a8/4a/3afe6a1efaf34461c18f5756a8eeb1b514e273cc8c79884732
Successfully built CryoAtom
22XP CryoAtom → /workspace/cryo-benchmark-work/runs/22XP/CryoAtom/run-k6bb9vnx/run.log
32MC CryoAtom → /workspace/cryo-benchmark-work/runs/32MC/CryoAtom/run-mqt56clc/run.log


In [7]:
# EMProt
emprot = checkout("EMProt", "https://github.com/huang-laboratory/EMProt.git", "9489d52")
emprot_run = conda_environment(emprot, "environment.yml")
if not (ENVS / "EMProt" / "bin" / "emprot").is_file():
    run([*emprot_run, "python", "-m", "pip", "install", "-e", "."], cwd=emprot)
if not (emprot / "emprot" / "weights" / "model_all_atom").is_dir():
    unpack("http://huanglab.phys.hust.edu.cn/EMProt/pretrained_weights/weights.tgz",
           "emprot.tar.gz", emprot / "emprot")

# EMProt takes a single chain per template file.
extract_chain = '''
import sys
from Bio.PDB import MMCIFParser, PDBIO
structure = MMCIFParser(QUIET=True).get_structure("af3", sys.argv[1])
writer = PDBIO()
writer.set_structure(next(structure.get_chains()))
writer.save(sys.argv[2])
'''


def build_emprot(target, attempt, log):
    template = attempt / "af3_chain.pdb"
    out = attempt / "model"
    run([*emprot_run, "python", "-c", extract_chain, target["af3"], template], cwd=emprot, log=log)
    run([*emprot_run, "emprot", "build", "--map", target["map"], "--seq", target["fasta"],
         "--chain", template, "--device", "0", "--output", out], cwd=emprot, log=log)
    return out / "output.cif"


benchmark("EMProt", build_emprot)

Cloning into '/workspace/cryo-benchmark-work/sources/tmpfmh8u7iu/EMProt'...


Channels:
 - defaults
 - conda-forge
 - pytorch
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /workspace/cryo-benchmark-work/envs/EMProt

  added / updated specs:
    - beartype==0.18.5
    - biopython==1.84
    - cctbx-base==2023.11
    - einops==0.8.0
    - jaxtyping==0.2.33
    - mrcfile==1.4.3
    - numba==0.59.1
    - numpy==1.26.2
    - pillow==10.0.1
    - pip=23.3.1
    - python=3.10.13
    - scipy==1.11.4
    - tqdm==4.66.5


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _libgcc_mutex-0.1          |             main           3 KB
    _openmp_mutex-5.1          |           52_gnu           7 KB
    beartype-0.18.5            |     pyhd8ed1ab_0         749 KB  conda-forge
    biopython-1.84             |  py310hc51659f_0         2.7 MB  conda-forge
    blas-1.0                   |         openblas          46 KB
    brot

WARNING conda.conda_pypi.main:notify_conda_pypi_tip(158): 
  Did you know? You can install many PyPI packages with conda
  using conda-pypi. Get started:
    https://bit.ly/4xXYt0B



done
Installing pip dependencies: ...working... Ran pip subprocess with arguments:
['/workspace/cryo-benchmark-work/envs/EMProt/bin/python', '-m', 'pip', 'install', '-U', '-r', '/workspace/cryo-benchmark-work/sources/EMProt/condaenv.t2gmgu7l.requirements.txt', '--exists-action=b']
Pip subprocess output:
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 5.2 MB/s eta 0:00:000:00:01m:020m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 11.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.1 MB/s eta 0:00:00a 0:00:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 8.1 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  164M  100  164M    0     0  2944k      0  0:00:57  0:00:57 --:--:-- 3088k


22XP EMProt → /workspace/cryo-benchmark-work/runs/22XP/EMProt/run-bpw3ew2w/run.log
32MC EMProt → /workspace/cryo-benchmark-work/runs/32MC/EMProt/run-876o798d/run.log


In [ ]:
# Score against the deposited structures (full assemblies).
install_phenix()
usalign = checkout("USalign", "https://github.com/pylelab/USalign.git", "23f14ff")
run(["make", "-C", usalign, "USalign"])

chain_comparison = '''
import json
import sys
from libtbx.utils import null_out
from mmtbx.validation.chain_comparison import run
scores = run(args=[sys.argv[1], sys.argv[2], "comparison.max_dist=3.0",
                   "input_files.unique_part_of_target_only=False",
                   "input_files.test_unique_part_of_target_only=False",
                   "crystal_info.use_crystal_symmetry=False"], out=null_out())
matched = scores.get_values("close")[1]
print(json.dumps({"ca_match_pct": 100 * matched / scores.used_chain,
                  "ca_quality_pct": 100 * matched / scores.used_target,
                  "sequence_match_pct": scores.get_match_percent("close") if matched else 0.0}))
'''


def score(model, reference):
    text = run([usalign / "USalign", model, reference, "-mol", "prot", "-mm", "1",
                "-ter", "1", "-outfmt", "2"], capture_output=True).stdout
    aligned = next(csv.DictReader(text.splitlines(), delimiter="\t"))
    comparison = run(phenix("phenix.python", "-c", chain_comparison, reference, model), capture_output=True)
    return dict(tm_score=float(aligned["TM2"]), aligned_ca=int(aligned["Lali"]),
                sequence_identity=float(aligned["IDali"]),
                **json.loads(comparison.stdout.splitlines()[-1]))


rows = []
for name, target in targets.items():
    for method in METHODS:
        record = RUNS / name / method / "result.json"
        row = dict(target=name, method=method, status="missing")
        if record.is_file():
            row.update(json.loads(record.read_text()))
        if row["status"] == "ok":
            try:
                row.update(score(row["model"], target["reference"]))
            except (OSError, ValueError, KeyError, IndexError, StopIteration, subprocess.CalledProcessError) as error:
                row.update(status="scoring failed", error=getattr(error, "stderr", None) or str(error))
        rows.append(row)

fields = ("target", "method", "status", "tm_score", "aligned_ca", "sequence_identity",
          "ca_match_pct", "ca_quality_pct", "sequence_match_pct", "runtime_seconds", "model", "log", "error")
csv_path = RUNS / "benchmark.csv"
with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
print(csv_path)